In [26]:
import pandas as pd
import torch 
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [27]:
live = pd.read_csv("../data/samples/trial/live_metrics.csv")
verbose = pd.read_csv("../data/samples/trial/verbose_statements.csv")
initial = pd.read_csv("../data/samples/trial/initial_statement.csv")

In [35]:
live = live.drop(columns=['Unnamed: 0'])

In [42]:
response_time = verbose['total_duration'].tolist()
response_time = response_time[:-1]
reset_iters = live[live['Iteration'] == 1].index.tolist()

gpu_util_avg = []
memory_util_avg = []
clock_util_avg = []

for i in range(1, len(reset_iters)):
    temp_metrics = live.dropna()
    gpu_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 3].tolist()
    memeory_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 9].tolist()
    clock_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], -2].tolist()
    gpu_util_avg.append(np.average(gpu_util_prompt))
    memory_util_avg.append(np.average(memeory_util_prompt))
    clock_util_avg.append(np.average(clock_util_prompt))

In [44]:
for (x, y, z) in zip(gpu_util_avg, memory_util_avg, clock_util_avg):
    print([x, y, z])

[0.0, 0.0, 0.09782608695652172]
[0.0, 0.0, 0.0978260869565217]
[0.0, 0.0, 0.09782608695652172]
[0.02, 0.02, 0.5489130434782609]
[0.04, 0.03, 1.0]
[0.04000000000000001, 0.030000000000000006, 1.0]
[0.7953550863723609, 0.524299424184261, 1.0]
[0.8264227642276425, 0.5499999999999998, 1.0]
[0.8254285714285716, 0.5597142857142858, 1.0]
[0.8292134831460676, 0.5700000000000002, 1.0]
[0.84, 0.5700000000000001, 1.0]
[0.8400000000000001, 0.5700000000000001, 1.0]
[0.84, 0.5700000000000001, 1.0]
[0.8149999999999998, 0.5583333333333332, 1.0]
[0.79, 0.55, 1.0]
[0.79, 0.55, 1.0]
[0.7985714285714287, 0.532857142857143, 1.0]
[0.8000000000000002, 0.5299999999999999, 1.0]
[0.8000000000000002, 0.5218421052631579, 1.0]
[0.8027272727272728, 0.5347727272727274, 1.0]
[0.8100000000000002, 0.5700000000000001, 1.0]
[0.81, 0.5700000000000001, 1.0]
[0.8100000000000002, 0.5700000000000001, 1.0]
[0.8099999999999999, 0.5615384615384615, 1.0]
[0.8099999999999998, 0.5500000000000002, 1.0]
[0.8175000000000001, 0.55769230

In [30]:
class BenchMark(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.l1 = nn.Sequential(
            nn.Linear(input_features, 32),
            nn.ReLU()
        )
        self.l2 = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU()
        )
        self.l3 = nn.Linear(16, output_features)

    def foward(self, x):
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        return x

In [31]:
model = BenchMark(4, 3)

In [45]:
def minimize_avg_time_loss(predicted, nums):
    total_sum = 0
    for i in range(3):
        total_sum += (predicted[i]+nums[i])
    return nn.MSELoss(total_sum, nums[-1]) 